# Step 3: Model Training & Segmentation
This notebook covers the dual-objective of performing **unsupervised segmentation** (K-Means) and **supervised poverty classification** (LightGBM). The logic is self-contained for easy experimentation.

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import joblib
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb

## 1. Household Segmentation (K-Means)
We group households into natural clusters based on their 142+ socio-economic features.

In [18]:
def train_segmentation(df, n_clusters=4):
    """Perform K-Means clustering for household segmentation."""
    features = df.drop(columns=['Id', 'idhogar', 'Target'], errors='ignore')
    
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features.fillna(0))
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(scaled_features)
    
    return kmeans, scaler, clusters

PROCESSED_PATH = '../data/processed/'
train_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'train_engineered.csv'))

N_CLUSTERS = 4
kmeans, scaler, clusters = train_segmentation(train_df, n_clusters=N_CLUSTERS)
train_df['Cluster'] = clusters

print(f"Successfully created {N_CLUSTERS} clusters.")

Successfully created 4 clusters.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.13/lib/p

## 2. Cluster Validation: Distribution Analysis
Comparing the predicted clusters with the original **Target** help us understand if our natural groupings align with socio-economic reality.

In [19]:
# Create a cross-tabulation of Cluster vs Target
cluster_target_dist = pd.crosstab(train_df['Cluster'], train_df['Target'], normalize='index') * 100

# Visualize as a stacked bar chart
fig = px.bar(cluster_target_dist, 
             title='Poverty Level Distribution within Each Cluster (%)',
             labels={'Cluster': 'User Segment (Cluster ID)', 'value': 'Percentage (%)'},
             barmode='stack')
fig.show()

print("Interpretation: If a cluster is dominated by Target 1 or 2, it indicates a high-risk household profile.")

Interpretation: If a cluster is dominated by Target 1 or 2, it indicates a high-risk household profile.


## 3. Poverty Classification (LightGBM)
We train a Gradient Boosting model to predict the official 4-level poverty scale.

### How LightGBM Works:
LightGBM is a **Gradient Boosting Decision Tree (GBDT)** framework. It builds thousands of small trees one by one. Each new tree focuses on the "residuals" (errors) of the previous ones. It is "Light" because it uses **Histograms** to bin continuous data, making it much faster than traditional XG-Boost.

### Understanding the F1 Score:
In poverty prediction, we care about two things:
1.  **Precision**: "Of everyone we *said* was in poverty, how many actually were?" (Avoiding wasted resources on wealthy homes).
2.  **Recall**: "Of everyone *actually* in poverty, how many did we find?" (Avoiding leaving families behind).

The **F1 Score** is the "harmonic mean"—it forces the model to be good at both. If you have infinite precision but 0 recall, your F1 score will be 0.

### Why "Macro" F1?
Our dataset is **Imbalanced** (there are many more wealthy households than extreme poverty ones). A normal accuracy score would be high just by guessing "Non-vulnerable" for everyone. **Macro F1** treats "Extreme Poverty" as just as important as "Non-poverty," even if we have fewer examples of it.

In [20]:
def train_classification(df):
    """Train a LightGBM classifier to predict poverty levels."""
    X = df.drop(columns=['Id', 'idhogar', 'Target', 'Cluster'], errors='ignore')
    y = df['Target'] - 1 # 0-indexed
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    params = {
        'objective': 'multiclass',
        'num_class': 4,
        'metric': 'multi_logloss',
        'learning_rate': 0.05,
        'verbosity': -1,
        'seed': 42
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
    
    model = lgb.train(params, train_data, num_boost_round=1000, 
                      valid_sets=[train_data, valid_data], 
                      callbacks=[lgb.early_stopping(stopping_rounds=50)])
    
    y_pred = np.argmax(model.predict(X_test), axis=1)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    return model, f1

model, f1_score_val = train_classification(train_df)
print(f"Macro F1 Score: {f1_score_val:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[32]	training's multi_logloss: 0.498583	valid_1's multi_logloss: 0.834412
Macro F1 Score: 0.3723


## 4. Saving Artifacts

In [21]:
MODEL_PATH = '../models/'
os.makedirs(MODEL_PATH, exist_ok=True)
joblib.dump(kmeans, os.path.join(MODEL_PATH, 'kmeans_model.joblib'))
joblib.dump(scaler, os.path.join(MODEL_PATH, 'scaler.joblib'))
model.save_model(os.path.join(MODEL_PATH, 'lgbm_model.txt'))
print(f"Models saved to {MODEL_PATH}")

Models saved to ../models/
